## Claude.ai -- Updates!

Push VQA to Claude.

### Docs:

* in theory, there is documentation here: https://docs.anthropic.com/en/home, 
* but in reality I just asked "Hey Claude.  How do I use your API through Python to upload an image and ask you questions about it?" followed by "Is there anyway to set the context?  For example, in chatgpt you have the ""role": "system", "content"" part of your message where you would say things like "You are a helpful assistant in charge of automating a process".  Or does one just incorporate that into the "question" part of the inputs?" and starting building from there

#### Key Points (cp-claude)

* API Key: Get your API key from the [Anthropic Console](https://console.anthropic.com/) and set it as an environment variable or pass it directly
* Supported formats: JPEG, PNG, GIF, and WebP images
* Model: Use `claude-sonnet-4-20250514` for Claude Sonnet 4
* Size limits: Images should be under 5MB and no larger than 8000x8000 pixels



First, install anthropic api (also, see .yml file for the environment for this project)

In [6]:
#!pip install anthropic

Where are things stored/going to be stored?

In [7]:
dir_api_old = '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/LMM_outputs/claude_api/' #store API results for example data
dir_api = '/Users/jnaiman/Downloads/tmp/iConf2025/tmpMultipanel/LMM_outputs/claude_api/' #store API results for example data

model ="claude-sonnet-4-20250514"

redo_file = '/Users/jnaiman/Downloads/tmp/iConf2025/tmpMultipanel/question_updates.json'

# where is VQA dataset?
jsons_dir = '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/jsons/' # directory where jsons created with figure are stored
imgs_dir = '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/imgs/' # where images are stored

# other stuff, where stored?
key_file = '/Users/jnaiman/.claudeai/key.txt'
# for saving temp images for reading in
tmp_dir = '/Users/jnaiman/Downloads/tmp/'

img_format = 'png'

In [8]:
import anthropic
import base64
from PIL import Image
import numpy as np
import json
import re
import pickle
import os
from glob import glob

# debug
from importlib import reload
from anthropic import RateLimitError


from sys import path
path.append('../')
import utils.llm_utils
reload(utils.llm_utils)
from utils.llm_utils import parse_qa, load_image, get_img_json_pair, parse_for_errors

import time
from utils.plot_qa_utils import get_nplots
from copy import deepcopy

In [9]:
# setup
with open(key_file,'r') as f:
    api_key = f.read()

client = anthropic.Anthropic(
  api_key=api_key.strip(),  # this is also the default, it can be omitted
)

In [10]:
jsons_to_parse = glob(jsons_dir + '/*.json')
jsons_to_parse[:3]

['/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/jsons/Picture186.json',
 '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/jsons/Picture169.json',
 '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/jsons/Picture56.json']

In [11]:
def send_to_claude(question_list, client, image_path, encoded_image,
                    model="claude-sonnet-4-20250514",
                    max_tokens=1000, temperature=0.1,
                    media_type = 'image/png',
                    tmp_dir = '/Users/jnaiman/Downloads/tmp/',
                    test_run = True, fac=1.0, 
                    verbose=True, verbose_tokens = False,
                    system_prompt = None, 
                    max_retries = 10, sleep_time=1):
    """
    Sends the question to claude and collects response.clear

    system_prompt : if None, then defaults to the overall system prompt generated with the questions.
    """
    if system_prompt is None:
        system_prompt = question_list['persona']

    #print('system prompt:', system_prompt)

    iFac = 2.0
    success = False
    #['persona', 'context','question', 'format']
    attempt = 0
    while not success and attempt < max_retries:
        try:
            question = question_list['context'] + " " + question_list['question'] + " " + question_list['format']
            # lowercase the first word, just in case
            question = question.lstrip() # no whitespace
            question = question[0].lower() + question[1:]
            if verbose: print('   on question:',question)
            # Prepare the API request
            prompt = f"I am going to show you an image. Now, {question}"
            prompt_save = f"I am going to show you an image. Now, {question}"
            ##question_list['prompt'] = prompt
            
            if not test_run:
                # Send the request to the Claude api
                response = client.messages.create(
                    model = model,
                    max_tokens=max_tokens,
                    system = system_prompt,
                    temperature=temperature,
                    messages=[
                        {
                            "role": "user",
                            "content": [
                                {
                                    "type": "image",
                                    "source": {
                                        "type": "base64",
                                        "media_type": media_type,
                                        "data": encoded_image,
                                    },
                                },
                                {
                                    "type": "text",
                                    "text": prompt
                                }
                            ],
                        }
                    ]
                )
                success = True
            else:
                success = True
        except RateLimitError as e:
            if attempt < max_retries - 1:
                # Exponential backoff with jitter
                wait_time = sleep_time*(2 ** (attempt)) + np.random.uniform(0, 1)
                print(f"      Rate limit hit. Waiting {wait_time:.2f} seconds before retry {attempt + 1}")
                time.sleep(wait_time)
                attempt += 1
            else:
                print(f"Max retries exceeded. Error: {e}")
                raise
        except Exception as e:
            print(e)
            new_fac = fac/iFac
            print('      new fac = ', new_fac)
            encoded_image = load_image(image_path,fac=new_fac, tmp_dir=tmp_dir)
            iFac += 1
    
    #print(response)
    if not test_run:
        # Get the response from the API
        answer = response.content[0].text #response.choices[0].message.content
        question_list['raw answer'] = answer
        # also calculate usage
        usage = response.usage
        question_list['usage'] = usage
        if verbose and verbose_tokens:
            print(f"      - Input tokens: {usage.input_tokens}")
            print(f"      - Output tokens: {usage.output_tokens}")
            print(f"      - Total tokens: {usage.input_tokens + usage.output_tokens}")
        # format answer
        answer_format = answer.split('```json"')[-1].split('\n')[0].replace('\n', '')
        #answer.replace("```json\n",'').replace("\n```",'')
        try:
            question_list['Response'] = json.loads(answer_format)
        except:
            question_list['Response'] = answer_format
            question_list['Error'] = 'JSON formatting'
        question_list['Response String'] = answer_format
        success = True
    else:
        question_list['Response'] = 'TEST RUN'
        question_list['Response String'] = 'TEST RUN'
        question_list['raw answer'] = 'TEST RUN'

    return question_list, prompt_save, system_prompt

In [12]:
with open(redo_file,'r') as f:
    redo_q_list = json.load(f)
    redo_q_list = json.loads(redo_q_list)

questions_redo = []
for q in redo_q_list:
    questions_redo.append(q['old'])

redo_q_list, questions_redo

([{'key': 'aspect ratio',
   'old': 'You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? You are a helpful assistant, please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.',
   'new': 'You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.',
   'new format': 'Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.',
   'new question': 'What is the aspect ratio of this figure?',
   'new context': ''},
  {'key': 'ylabels',
   'old': 'You are a helpful assistant that can analyze images.  What are the x-axis titles for each figure panel? Please format the output as a json as {"ylabels":[]}, where the list is a list of strings of all of the y-axis titles. If there is a single plot, this should be one element in this list, and if th

In [15]:
iMax = 2000
verbose = False
test_run = False # run w/o actually pinging openai
restart = False
max_tokens=500
# set system_prompt to None to default to what is in question list
system_prompt = """You are a helpful assistant that responds only in valid JSON format. Do not include any explanations, reasoning, or text outside of the JSON response."""
#system_prompt = """You must respond with only valid JSON. Start your response immediately with { and end with }. Do not write any text before or after the JSON."""
temperature=0.1
fac = 1.0
sleep = 5 # seconds
use_single_prompt = True # use 1 prompt and 1 image, if False will use multiple at a time

max_img_size = (8000,8000)

import utils.llm_utils
reload(utils.llm_utils)
from utils.llm_utils import get_img_json_pair

if test_run:
    sleep = 0.0

for ijson,json_path in enumerate(jsons_to_parse):
    if ijson >= iMax:
        continue

    print('on', ijson, 'of', min(iMax,len(jsons_to_parse)))

    # get image and base json
    img_path = imgs_dir + json_path.split('/')[-1].removesuffix('.json') + '.' + img_format
    print(img_path)
    encoded_image, img_format_media, base_json, err = get_img_json_pair(img_path, json_path, dir_api, 
                                                      fac=fac, restart=restart,
                                                      tmp_dir=tmp_dir, max_img_size=max_img_size)
    if err:
        continue

    ###### create QA ########

    # also load the already-filled json from prior run
    with open(dir_api_old + json_path.split('/')[-1].removesuffix('.json') + '.pickle', 'rb') as f:
        qa_orig, model = pickle.load(f)

    # find ones to rerun
    qa = []; redos = []
    for qa_pair in qa_orig:
        if qa_pair['Q'] in questions_redo:
            Qnew = redo_q_list[questions_redo.index(qa_pair['Q'])]
            print('** REDO:', qa_pair['Q'])
            qaout = deepcopy(qa_pair)
            # empty things
            for k in ['Response', 'raw answer', 'Response String', 'prompt']:
                qaout[k] = ""
            qaout['fac'] = 1.0
            # update question
            qaout['Q'] = Qnew['new']
            qaout['format'] = Qnew['new format']
            qaout['question'] = Qnew['new question']
            qa.append(qaout)
            redos.append(True)
            print('** NEW: ', qaout)
            #import sys; sys.exit()
        else:
            qa.append(deepcopy(qa_pair))
            redos.append(False)

    responses = []; prompts = []; system_prompts = []
    if use_single_prompt:
        for question_list,rd in zip(qa,redos):
            if rd:
                response, prompt, system_prompt_out = send_to_claude(question_list, client, img_path, encoded_image,
                            model = model, max_tokens=max_tokens, media_type='image/' + img_format_media,
                            test_run = test_run, system_prompt=system_prompt, temperature=temperature, 
                            sleep_time=sleep, fac=fac)
                responses.append(response)
                question_list['prompt'] = prompt
                question_list['system prompt'] = system_prompt_out
            else:
                responses.append(deepcopy(question_list['Response']))
        #import sys; sys.exit() # just do one
        time.sleep(sleep)
    time.sleep(sleep)

    # parse for errors
    #qa = parse_for_errors(qa, llm='claude')
    print('')
    print('**** Cleaned QA ****')
    #qa = parse_for_errors_claude(qa)
    qa = parse_for_errors(qa, llm='Claude')

    # dump to file
    if not test_run:
        with open(dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle', 'wb') as ff:
            pickle.dump([qa, model], ff)
        print('Just saved:', dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle')
    else:
        print('Would store at:', dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle')
    #import sys; sys.exit()

on 0 of 200
/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/imgs/Picture186.png
have file already: /Users/jnaiman/Downloads/tmp/iConf2025/tmpMultipanel/LMM_outputs/claude_api/Picture186.pickle
on 1 of 200
/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/imgs/Picture169.png
have file already: /Users/jnaiman/Downloads/tmp/iConf2025/tmpMultipanel/LMM_outputs/claude_api/Picture169.pickle
on 2 of 200
/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/imgs/Picture56.png
** REDO: You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? You are a helpful assistant, please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.
** NEW:  {'Q': 'You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.', 'A': {'aspect ratio': 1.4103896689207418}, 'Level': 'Level 

/opt/anaconda3/envs/JCDL2025/lib/python3.12/site-packages/PIL/Image.py:3406: DecompressionBombWarning: Image size (116848180 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


** REDO: You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? You are a helpful assistant, please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.
** NEW:  {'Q': 'You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.', 'A': {'aspect ratio': 2.423609267183984}, 'Level': 'Level 1', 'type': 'Figure-level questions', 'Response': '', 'persona': 'You are a helpful assistant that can analyze images.', 'context': '', 'question': 'What is the aspect ratio of this figure?', 'format': 'Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.', 'raw answer': '', 'usage': Usage(cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=1638, output_tokens=17, server_tool_use=None, service_tier='standard', cache_creation={'ephemera

/opt/anaconda3/envs/JCDL2025/lib/python3.12/site-packages/PIL/Image.py:3406: DecompressionBombWarning: Image size (111573300 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


** REDO: You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? You are a helpful assistant, please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.
** NEW:  {'Q': 'You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.', 'A': {'aspect ratio': 1.2292206201761557}, 'Level': 'Level 1', 'type': 'Figure-level questions', 'Response': '', 'persona': 'You are a helpful assistant that can analyze images.', 'context': '', 'question': 'What is the aspect ratio of this figure?', 'format': 'Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.', 'raw answer': '', 'usage': Usage(cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=1654, output_tokens=17, server_tool_use=None, service_tier='standard', cache_creation={'ephemer

...

** REDO: You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? You are a helpful assistant, please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.
** NEW:  {'Q': 'You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.', 'A': {'aspect ratio': 4.9380480016223665}, 'Level': 'Level 1', 'type': 'Figure-level questions', 'Response': '', 'persona': 'You are a helpful assistant that can analyze images.', 'context': '', 'question': 'What is the aspect ratio of this figure?', 'format': 'Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.', 'raw answer': '', 'usage': Usage(cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=1630, output_tokens=17, server_tool_use=None, service_tier='standard', cache_creation={'ephemer

/opt/anaconda3/envs/JCDL2025/lib/python3.12/site-packages/PIL/Image.py:3406: DecompressionBombWarning: Image size (111998125 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


** REDO: You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? You are a helpful assistant, please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.
** NEW:  {'Q': 'You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.', 'A': {'aspect ratio': 1.2401985128781283}, 'Level': 'Level 1', 'type': 'Figure-level questions', 'Response': '', 'persona': 'You are a helpful assistant that can analyze images.', 'context': '', 'question': 'What is the aspect ratio of this figure?', 'format': 'Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.', 'raw answer': '', 'usage': Usage(cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=1650, output_tokens=17, server_tool_use=None, service_tier='standard', cache_creation={'ephemer

/opt/anaconda3/envs/JCDL2025/lib/python3.12/site-packages/PIL/Image.py:3406: DecompressionBombWarning: Image size (152060850 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


** REDO: You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? You are a helpful assistant, please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.
** NEW:  {'Q': 'You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.', 'A': {'aspect ratio': 5.724328402933526}, 'Level': 'Level 1', 'type': 'Figure-level questions', 'Response': '', 'persona': 'You are a helpful assistant that can analyze images.', 'context': '', 'question': 'What is the aspect ratio of this figure?', 'format': 'Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.', 'raw answer': '', 'usage': Usage(cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=1644, output_tokens=17, server_tool_use=None, service_tier='standard', cache_creation={'ephemera

## Look at data

Check out one, if you wanna:

In [25]:
pickles = glob(dir_api + '*.pickle')
pickles[:5]

['/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/LMM_outputs/claude_api/Picture11.pickle',
 '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/LMM_outputs/claude_api/Picture182.pickle',
 '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/LMM_outputs/claude_api/Picture72.pickle',
 '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/LMM_outputs/claude_api/Picture190.pickle',
 '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/LMM_outputs/claude_api/Picture60.pickle']

In [26]:
ifile = 0
with open(pickles[ifile], 'rb') as f:
    qa_in = pickle.load(f)[0]

In [27]:
qa_in[0]

{'Q': 'You are a helpful assistant that can analyze images.  How many panels are in this figure? Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns.',
 'A': {'nrows': 3, 'ncols': 3},
 'Level': 'Level 1',
 'type': 'Figure-level questions',
 'Response': {'nrows': '3', 'ncols': '3'},
 'persona': 'You are a helpful assistant that can analyze images.',
 'context': '',
 'question': 'How many panels are in this figure?',
 'format': 'Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns.',
 'raw answer': '```json\n{"nrows": "3", "ncols": "3"}\n```',
 'usage': Usage(cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=1264, output_tokens=22, server_tool_use=None, service_tier='standard', cache_creation={'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}),
 'Response String': '```json',
 'prompt': 'I am going to show you an image. Now, how many panels are in this fi

Claude outputs reasoning, so we have to do a bit of cleaning from the responses:

In [28]:
print(pickles[ifile])
print('*********')
for qa_pairs in qa_in:
    print('Prompt:', qa_pairs['prompt'])
    print('  Real A:', qa_pairs['A'])
    print('Claude A:', qa_pairs['Response'])
    print('')

/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/LMM_outputs/claude_api/Picture11.pickle
*********
Prompt: I am going to show you an image. Now, how many panels are in this figure? Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns.
  Real A: {'nrows': 3, 'ncols': 3}
Claude A: {'nrows': '3', 'ncols': '3'}

Prompt: I am going to show you an image. Now, assume this is a figure made with matplotlib in Python.  Examples of plotting styles are "classic" or "ggplot". Examples of plotting styles are "classic" or "ggplot". What is the plot style used in this figure? Please format the output as a json as {"plot style":""} to store the matplotlib plotting style used in the figure.
  Real A: {'plot style': 'Solarize_Light2'}
Claude A: {'plot style': 'ggplot'}

Prompt: I am going to show you an image. Now, assume this is a figure made with matplotlib in Python. Examples of matplotlib colormaps are "rainbow" or "Reds". What is the color